### <u>NOTE</u>:
For the data:
- [Head here for the flight dataset](https://www.kaggle.com/datasets/hrishitpatil/flight-data-2024/data)
- [Head here for the airprots dataset](https://data.humdata.org/dataset/ourairports-usa)

Other note(s):
- This file also assumes you do <u>*NOT*</u> have a duckdb setup in your local of this repo already
- As usual with backend files, **run this from your /backend**
***
⚠️⚠️ Once they are both downloaded as .CSV's, move them to: `/backend/data/csv` of this project ⚠️⚠️
***

# **<u>CSV Data runner</u>**

### What this file does:
- Converts CSV's into duckdb tables
- Cleans that data efficiently using duckdb
- Creates new `weather_api` composite table, which shows all the queries needing to be made to the weather API

In [1]:
import sys
from pathlib import Path
import duckdb as ddb

def _find_proj_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "backend/pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing backend/pyproject.toml")


PROJ_ROOT = _find_proj_root(Path.cwd())

if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))

from backend.src.ml.csv_data_pipeline_funcs import (
    CsvTableBuildError,
    DataPipelineError,
    WeatherRequestTableBuildError,
    create_and_clean_airport_table,
    create_and_clean_flights_table,
    create_weather_req_table,
)

In [2]:
# Constants
DATA_DIR = PROJ_ROOT/"backend/data"

# .csv'
FLIGHT_DATA_CSV_PATH = DATA_DIR/"csv/flight_data_2024_sample.csv" # OR USE 'flight_data_2024.csv' for real set (1m+ rows. Ensure you have RAM)
AIRPORT_DATA_CSV_PATH = DATA_DIR/"csv/us-airports.csv"

# table names
AIRPORT_TABLE_NAME = "airport_data"
FLIGHT_TABLE_NAME = "flight_data_sample"

# Open the connection and create the duckdb
DUCK_DB_PATH = DATA_DIR/"duck_database.duckdb"

In [3]:
# Write to both tables
try:
    con = ddb.connect(DUCK_DB_PATH)
    f_response = create_and_clean_flights_table(con, FLIGHT_TABLE_NAME, FLIGHT_DATA_CSV_PATH)
    print(f_response.message)
    display(f_response.data)

    a_response = create_and_clean_airport_table(con, AIRPORT_TABLE_NAME, AIRPORT_DATA_CSV_PATH)
    print(a_response.message)
    display(a_response.data)

    wrt_response = create_weather_req_table(con, flight_table_name=FLIGHT_TABLE_NAME, airport_table_name=AIRPORT_TABLE_NAME)
    print(wrt_response.message)
    display(wrt_response.data)
    
except (CsvTableBuildError, WeatherRequestTableBuildError) as exc:
    print(f"Pipeline stage failed: {exc}")
    raise
except DataPipelineError as exc:
    print(f"Pipeline failed: {exc}")
    raise
except Exception as exc:
    print(f"Unexpected failure: {exc}")
    raise
finally:
    con.close()

Successfully created and cleaned flight data under table name: flight_data_sample


[{'column_name': 'date',
  'column_type': 'DATE',
  'null': 'YES',
  'key': None,
  'default': None,
  'extra': None},
 {'column_name': 'flight_number',
  'column_type': 'DOUBLE',
  'null': 'YES',
  'key': None,
  'default': None,
  'extra': None},
 {'column_name': 'origin',
  'column_type': 'VARCHAR',
  'null': 'YES',
  'key': None,
  'default': None,
  'extra': None},
 {'column_name': 'origin_city_name',
  'column_type': 'VARCHAR',
  'null': 'YES',
  'key': None,
  'default': None,
  'extra': None},
 {'column_name': 'dest',
  'column_type': 'VARCHAR',
  'null': 'YES',
  'key': None,
  'default': None,
  'extra': None},
 {'column_name': 'dest_city_name',
  'column_type': 'VARCHAR',
  'null': 'YES',
  'key': None,
  'default': None,
  'extra': None},
 {'column_name': 'pred_dep_time',
  'column_type': 'BIGINT',
  'null': 'YES',
  'key': None,
  'default': None,
  'extra': None},
 {'column_name': 'pred_arr_time',
  'column_type': 'BIGINT',
  'null': 'YES',
  'key': None,
  'default': Non

Successfully created and cleaned airport data under table name: airport_data


[{'column_name': 'name',
  'column_type': 'VARCHAR',
  'null': 'YES',
  'key': None,
  'default': None,
  'extra': None},
 {'column_name': 'lat',
  'column_type': 'DOUBLE',
  'null': 'YES',
  'key': None,
  'default': None,
  'extra': None},
 {'column_name': 'long',
  'column_type': 'DOUBLE',
  'null': 'YES',
  'key': None,
  'default': None,
  'extra': None},
 {'column_name': 'code',
  'column_type': 'VARCHAR',
  'null': 'YES',
  'key': None,
  'default': None,
  'extra': None}]

Successfully created 'weather_req_table'


[{'column_name': 'date',
  'column_type': 'DATE',
  'null': 'YES',
  'key': None,
  'default': None,
  'extra': None},
 {'column_name': 'code',
  'column_type': 'VARCHAR',
  'null': 'YES',
  'key': None,
  'default': None,
  'extra': None},
 {'column_name': 'name',
  'column_type': 'VARCHAR',
  'null': 'YES',
  'key': None,
  'default': None,
  'extra': None},
 {'column_name': 'lat',
  'column_type': 'DOUBLE',
  'null': 'YES',
  'key': None,
  'default': None,
  'extra': None},
 {'column_name': 'long',
  'column_type': 'DOUBLE',
  'null': 'YES',
  'key': None,
  'default': None,
  'extra': None},
 {'column_name': 'status',
  'column_type': 'VARCHAR',
  'null': 'YES',
  'key': None,
  'default': None,
  'extra': None},
 {'column_name': 'attempt_count',
  'column_type': 'INTEGER',
  'null': 'YES',
  'key': None,
  'default': None,
  'extra': None},
 {'column_name': 'last_error',
  'column_type': 'VARCHAR',
  'null': 'YES',
  'key': None,
  'default': None,
  'extra': None},
 {'column_nam